In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
from collections import deque
import random
import copy
GOAL_STATE = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]
ACTIONS = {'U': (-1, 0), 'D': (1, 0), 'L': (0, -1), 'R': (0, 1)}
ACTION_NAMES = {'U': 'Up', 'D': 'Down', 'L': 'Left', 'R': 'Right'}
BG_DARK       = "#0f0f1a"
BG_CARD       = "#1a1a2e"
BG_CARD_ALT   = "#16213e"
ACCENT_BLUE   = "#0f3460"
ACCENT_CYAN   = "#00d2ff"
ACCENT_PURPLE = "#7b2ff7"
ACCENT_PINK   = "#e94560"
ACCENT_GREEN  = "#00e676"
ACCENT_ORANGE = "#ff9100"
TEXT_PRIMARY   = "#e0e0e0"
TEXT_SECONDARY = "#9e9e9e"
TEXT_BRIGHT    = "#ffffff"
TILE_COLORS   = [
    "#2c2c3e",   # 0 - empty
    "#e94560",   # 1
    "#ff6b6b",   # 2
    "#ffa726",   # 3
    "#ffee58",   # 4
    "#66bb6a",   # 5
    "#26c6da",   # 6
    "#42a5f5",   # 7
    "#ab47bc",   # 8
]
TILE_TEXT_COLORS = [
    "#2c2c3e",   # 0
    "#ffffff",   # 1
    "#ffffff",   # 2
    "#1a1a2e",   # 3
    "#1a1a2e",   # 4
    "#1a1a2e",   # 5
    "#1a1a2e",   # 6
    "#ffffff",   # 7
    "#ffffff",   # 8
]

class Node:
    """
    Đại diện cho một node trong cây tìm kiếm.
    - state: ma trận 3x3 (list of list)
    - action: hành động từ node cha ('L','R','U','D' hoặc None)
    - depth: số bước (hành động) đã thực hiện từ trạng thái ban đầu
    - parent: node cha (Node hoặc None)
    """
    def __init__(self, state, action=None, depth=0, parent=None):
        self.state = state
        self.action = action
        self.depth = depth
        self.parent = parent

    def state_tuple(self):
        return tuple(tuple(row) for row in self.state)

    def __repr__(self):
        action_str = self.action if self.action else "—"
        return f"Action={action_str}, Depth={self.depth}"

def find_blank(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
    return None

def get_children(node):
    """Sinh ra tất cả các node con từ node hiện tại."""
    children = []
    bi, bj = find_blank(node.state)
    for action, (di, dj) in ACTIONS.items():
        ni, nj = bi + di, bj + dj
        if 0 <= ni < 3 and 0 <= nj < 3:
            new_state = copy.deepcopy(node.state)
            new_state[bi][bj], new_state[ni][nj] = new_state[ni][nj], new_state[bi][bj]
            child = Node(new_state, action, node.depth + 1, node)
            children.append(child)
    return children


def h(node):
    """
    Hàm đánh giá Greedy: đếm số ô SAI vị trí so với GOAL_STATE.
    Ô trống (0) không tính.
    """
    count = 0
    for i in range(3):
        for j in range(3):
            val = node.state[i][j]
            if val != 0 and val != GOAL_STATE[i][j]:
                count += 1
    return count

def is_solvable(state):
    flat = [x for row in state for x in row if x != 0]
    inversions = 0
    for i in range(len(flat)):
        for j in range(i + 1, len(flat)):
            if flat[i] > flat[j]:
                inversions += 1
    return inversions % 2 == 0

def generate_random_state():
    while True:
        nums = list(range(9))
        random.shuffle(nums)
        state = [nums[i*3:(i+1)*3] for i in range(3)]
        if is_solvable(state) and state != GOAL_STATE:
            return state

class StepRecord:
    """Lưu trạng thái tại mỗi bước của thuật toán."""
    def __init__(self, current_node, frontier_list, reached_set, description=""):
        self.current_node = current_node
        self.frontier_list = list(frontier_list)
        self.reached_set = set(reached_set)
        self.description = description

def solve_steps_greedy(initial_state):
    """
    Greedy Best-First Search:
    - Hàm đánh giá h(node) = số ô SAI vị trí trong trạng thái của node.
    - Mỗi lần sinh con, tính h() cho từng con rồi chọn node có h() nhỏ nhất
      đưa vào đầu frontier (ưu tiên xét trước).
    - Frontier luôn được sắp xếp theo h() tăng dần.
    """
    steps = []
    root = Node(initial_state, None, 0, None)
    root.h_val = h(root) 

    frontier = [root]
    reached = set()

    steps.append(StepRecord(
        current_node=None,
        frontier_list=[root],
        reached_set=set(),
        description=f"Khởi tạo: Frontier = [root], h(root) = {root.h_val}"
    ))

    max_steps = 5000

    while frontier and len(steps) < max_steps:
        current = frontier.pop(0)
        state_t = current.state_tuple()

        if state_t in reached:
            continue

        reached.add(state_t)

        steps.append(StepRecord(
            current_node=current,
            frontier_list=list(frontier),
            reached_set=set(reached),
            description=(
                f"Xét node (depth={current.depth}, action={current.action or 'Start'}, "
                f"h={current.h_val})"
            )
        ))

        if current.state == GOAL_STATE:
            steps.append(StepRecord(
                current_node=current,
                frontier_list=list(frontier),
                reached_set=set(reached),
                description="✅ ĐÃ TÌM THẤY GOAL!"
            ))
            return steps

        children = get_children(current)
        new_children = []
        for child in children:
            if child.state_tuple() not in reached:
                child.h_val = h(child)
                frontier.append(child)
                new_children.append(child)

        frontier.sort(key=lambda nd: nd.h_val)

        h_vals = [c.h_val for c in new_children]
        steps.append(StepRecord(
            current_node=current,
            frontier_list=list(frontier),
            reached_set=set(reached),
            description=(
                f"Sinh {len(new_children)} node con → h={h_vals} → "
                f"Sắp xếp Frontier theo h() tăng dần"
            )
        ))

    if len(steps) >= max_steps:
        steps.append(StepRecord(
            current_node=None,
            frontier_list=[],
            reached_set=reached,
            description="⚠️ Đã đạt giới hạn bước mô phỏng."
        ))

    return steps


def solve_steps_astar(initial_state):
    """
    A* Search:
    - f(n) = g(n) + h(n)
    - g(n) = node.depth (số hành động từ đầu đến n)
    - h(n) = số ô SAI vị trí (misplaced tiles)
    - Frontier được sắp xếp theo f(n) tăng dần.
    """
    steps = []
    root = Node(initial_state, None, 0, None)
    root.h_val = h(root)
    root.f_val = root.h_val + root.depth  

    frontier = [root]
    reached = set()

    steps.append(StepRecord(
        current_node=None,
        frontier_list=[root],
        reached_set=set(),
        description=(
            f"Khởi tạo: h={root.h_val}, g={root.depth}, "
            f"f={root.f_val}"
        )
    ))

    max_steps = 5000

    while frontier and len(steps) < max_steps:
        current = frontier.pop(0)
        state_t = current.state_tuple()

        if state_t in reached:
            continue

        reached.add(state_t)

        steps.append(StepRecord(
            current_node=current,
            frontier_list=list(frontier),
            reached_set=set(reached),
            description=(
                f"Xét node: g={current.depth}, "
                f"h={current.h_val}, f={current.f_val} "
                f"(action={current.action or 'Start'})"
            )
        ))

        if current.state == GOAL_STATE:
            steps.append(StepRecord(
                current_node=current,
                frontier_list=list(frontier),
                reached_set=set(reached),
                description="✅ ĐÃ TÌM THẤY GOAL! (A*)"
            ))
            return steps

        children = get_children(current)
        new_children = []
        for child in children:
            if child.state_tuple() not in reached:
                child.h_val = h(child)
                child.f_val = child.depth + child.h_val
                frontier.append(child)
                new_children.append(child)

        frontier.sort(key=lambda nd: nd.f_val)

        f_vals = [c.f_val for c in new_children]
        steps.append(StepRecord(
            current_node=current,
            frontier_list=list(frontier),
            reached_set=set(reached),
            description=(
                f"Sinh {len(new_children)} con → f={f_vals} → "
                f"Sắp xếp Frontier theo f()=g+h"
            )
        ))

    if len(steps) >= max_steps:
        steps.append(StepRecord(
            current_node=None,
            frontier_list=[],
            reached_set=reached,
            description="⚠️ Đã đạt giới hạn bước mô phỏng."
        ))

    return steps


def solve_steps_ids(initial_state):
    """
    Iterative Deepening Search (IDS):
    - Khởi tạo depth limit = 0.
    - Chạy DFS giới hạn độ sâu (DLS). Nếu không thấy, tăng limit lên 1 và lặp lại.
    """
    steps = []
    max_total_steps = 5000
    
    for limit in range(100):
        root = Node(initial_state, None, 0, None)
        frontier = [root]
        reached = {root.state_tuple(): 0}
        
        steps.append(StepRecord(
            current_node=None,
            frontier_list=[root],
            reached_set=set(reached.keys()),
            description=f"--- Bắt đầu IDS với Limit = {limit} ---"
        ))
        
        while frontier and len(steps) < max_total_steps:
            current = frontier.pop()
            
            steps.append(StepRecord(
                current_node=current,
                frontier_list=list(frontier),
                reached_set=set(reached.keys()),
                description=f"Xét node (depth={current.depth}/{limit}, action={current.action or 'Start'})"
            ))
            
            if current.state == GOAL_STATE:
                steps.append(StepRecord(
                    current_node=current,
                    frontier_list=list(frontier),
                    reached_set=set(reached.keys()),
                    description=f"✅ ĐÃ TÌM THẤY GOAL ở limit {limit}!"
                ))
                return steps
                
            if current.depth < limit:
                children = get_children(current)
                new_children = []
                for child in children:
                    child_t = child.state_tuple()
                    if child_t not in reached or reached[child_t] > child.depth:
                        reached[child_t] = child.depth
                        frontier.append(child)
                        new_children.append(child)
                
                steps.append(StepRecord(
                    current_node=current,
                    frontier_list=list(frontier),
                    reached_set=set(reached.keys()),
                    description=f"Sinh {len(new_children)} node con (còn hạn mức depth={limit})"
                ))
            else:
                steps.append(StepRecord(
                    current_node=current,
                    frontier_list=list(frontier),
                    reached_set=set(reached.keys()),
                    description=f"Đạt giới hạn depth={limit}, không sinh con."
                ))
                
        if len(steps) >= max_total_steps:
            steps.append(StepRecord(
                current_node=None,
                frontier_list=[],
                reached_set=set(reached.keys()),
                description="⚠️ Đã đạt giới hạn bước mô phỏng tổng cộng."
            ))
            return steps
            
    return steps


def solve_steps_lddg(initial_state):
    """
    Leo đồi đơn giản (LĐĐG):
    - h(node) = số ô SAI vị trí.
    - Lần lượt sinh các con. Node con nào sinh ra sẽ được đưa vào frontier.
    - Nếu h(con) < h(node hiện tại): lấy làm node xét tiếp theo (frontier bị clear).
    - Nếu h(con) >= h(node hiện tại): loại ra khỏi frontier.
    - Dừng nếu đã thử hết các con mà không có con nào tốt hơn.
    """
    steps = []
    current = Node(initial_state, None, 0, None)
    current.h_val = h(current)
    
    reached = set()
    frontier = []
    
    steps.append(StepRecord(
        current_node=None,
        frontier_list=[current],
        reached_set=set(),
        description=f"Khởi tạo: h(root) = {current.h_val}"
    ))
    
    max_steps = 5000
    
    while current and len(steps) < max_steps:
        state_t = current.state_tuple()
        reached.add(state_t)
        
        # Thêm current vào frontier để hiển thị ở bước hiện tại
        if not frontier:
            frontier = [current]
            
        steps.append(StepRecord(
            current_node=current,
            frontier_list=list(frontier),
            reached_set=set(reached),
            description=(
                f"Xét node (depth={current.depth}, action={current.action or 'Start'}, "
                f"h={current.h_val})"
            )
        ))
        
        if current.state == GOAL_STATE:
            steps.append(StepRecord(
                current_node=current,
                frontier_list=list(frontier),
                reached_set=set(reached),
                description="✅ ĐÃ TÌM THẤY GOAL!"
            ))
            return steps
            
        children = get_children(current)
        found_better = False
        frontier.clear()
        
        for child in children:
            child.h_val = h(child)
            frontier.append(child)
            
            steps.append(StepRecord(
                current_node=current,
                frontier_list=list(frontier),
                reached_set=set(reached),
                description=f"Sinh node con (action={child.action}), h={child.h_val} → Đưa vào frontier."
            ))
            
            if child.h_val < current.h_val:
                steps.append(StepRecord(
                    current_node=current,
                    frontier_list=list(frontier),
                    reached_set=set(reached),
                    description=f"Node con có h={child.h_val} < {current.h_val} (node hiện tại). Lấy làm node xét tiếp theo!"
                ))
                current = child
                frontier = [child]
                found_better = True
                break
            else:
                frontier.pop()
                steps.append(StepRecord(
                    current_node=current,
                    frontier_list=list(frontier),
                    reached_set=set(reached),
                    description=f"Node con có h={child.h_val} >= {current.h_val}. Loại ra khỏi frontier."
                ))
                
        if not found_better:
            steps.append(StepRecord(
                current_node=current,
                frontier_list=list(frontier),
                reached_set=set(reached),
                description="Không có node con nào có h() nhỏ hơn node hiện tại. Dừng leo đồi."
            ))
            return steps

    if len(steps) >= max_steps:
        steps.append(StepRecord(
            current_node=None,
            frontier_list=list(frontier),
            reached_set=reached,
            description="⚠️ Đã đạt giới hạn bước mô phỏng."
        ))

    return steps


def solve_steps_lddn(initial_state):
    """
    Leo đồi dốc nhất (LĐDN):
    - h(node) = số ô SAI vị trí.
    - Sinh ra tất cả các con.
    - So sánh và chọn node con có h() nhỏ nhất.
    - Nếu h(con nhỏ nhất) < h(node hiện tại): lấy làm gốc tiếp theo.
    - Dừng nếu không có con nào có h() < h(node hiện tại) hoặc tìm thấy Goal.
    """
    steps = []
    current = Node(initial_state, None, 0, None)
    current.h_val = h(current)
    
    reached = set()
    frontier = []
    
    steps.append(StepRecord(
        current_node=None,
        frontier_list=[current],
        reached_set=set(),
        description=f"Khởi tạo: h(root) = {current.h_val}"
    ))
    
    max_steps = 5000
    
    while current and len(steps) < max_steps:
        state_t = current.state_tuple()
        reached.add(state_t)
        
        if not frontier:
            frontier = [current]
            
        steps.append(StepRecord(
            current_node=current,
            frontier_list=list(frontier),
            reached_set=set(reached),
            description=(
                f"Xét node (depth={current.depth}, action={current.action or 'Start'}, "
                f"h={current.h_val})"
            )
        ))
        
        if current.state == GOAL_STATE:
            steps.append(StepRecord(
                current_node=current,
                frontier_list=list(frontier),
                reached_set=set(reached),
                description="✅ ĐÃ TÌM THẤY GOAL!"
            ))
            return steps
            
        children = get_children(current)
        best_child = None
        best_h = float('inf')
        
        frontier.clear()
        for child in children:
            child.h_val = h(child)
            frontier.append(child)
            if child.h_val < best_h:
                best_h = child.h_val
                best_child = child
                
        h_vals = [c.h_val for c in children]
        steps.append(StepRecord(
            current_node=current,
            frontier_list=list(frontier),
            reached_set=set(reached),
            description=f"Sinh tất cả {len(children)} node con → h={h_vals} → Đưa vào frontier để so sánh."
        ))
        
        if best_child and best_child.h_val < current.h_val:
            frontier = [best_child]
            steps.append(StepRecord(
                current_node=current,
                frontier_list=list(frontier), 
                reached_set=set(reached),
                description=(
                    f"Chọn node con có h() nhỏ nhất (h={best_child.h_val} < {current.h_val}) làm gốc tiếp theo. "
                    f"Xóa các node con khác khỏi frontier."
                )
            ))
            current = best_child
        else:
            frontier.clear()
            steps.append(StepRecord(
                current_node=current,
                frontier_list=list(frontier),
                reached_set=set(reached),
                description=(
                    f"Không có con nào h() < {current.h_val} (nhỏ nhất là {best_child.h_val if best_child else 'N/A'}). "
                    f"Xóa các node con khỏi frontier và dừng leo đồi."
                )
            ))
            return steps

    if len(steps) >= max_steps:
        steps.append(StepRecord(
            current_node=None,
            frontier_list=list(frontier),
            reached_set=reached,
            description="⚠️ Đã đạt giới hạn bước mô phỏng."
        ))

    return steps


def solve_steps(initial_state, algorithm='BFS c1'):
    """
    Chạy thuật toán và lưu lại từng bước.
    Trả về list các StepRecord.
    """
    steps = []
    root = Node(initial_state, None, 0, None)

    if 'LĐĐG' in algorithm:
        return solve_steps_lddg(initial_state)

    if 'LĐDN' in algorithm:
        return solve_steps_lddn(initial_state)

    if 'IDS' in algorithm:
        return solve_steps_ids(initial_state)

    if 'Greedy' in algorithm:
        return solve_steps_greedy(initial_state)

    if 'A*' in algorithm:
        return solve_steps_astar(initial_state)

    is_bfs = 'BFS' in algorithm
    is_c2 = 'c2' in algorithm

    if is_bfs:
        frontier = deque([root])
    else: 
        frontier = [root]

    reached = set()

    steps.append(StepRecord(
        current_node=None,
        frontier_list=[root],
        reached_set=set(),
        description="Khởi tạo: Frontier chứa trạng thái ban đầu"
    ))

    if is_c2 and root.state == GOAL_STATE:
        steps.append(StepRecord(
            current_node=root,
            frontier_list=list(frontier),
            reached_set=set(),
            description="✅ ĐÃ TÌM THẤY GOAL ở trạng thái ban đầu!"
        ))
        return steps

    max_steps = 5000  # giới hạn để tránh treo

    while frontier and len(steps) < max_steps:
        if is_bfs:
            current = frontier.popleft()
        else:
            current = frontier.pop()

        state_t = current.state_tuple()

        if state_t in reached:
            continue

        reached.add(state_t)

        # Ghi bước: đang xét node
        steps.append(StepRecord(
            current_node=current,
            frontier_list=list(frontier),
            reached_set=set(reached),
            description=f"Xét node (depth={current.depth}, action={current.action or 'Start'})"
        ))

        # Đối với c1: Kiểm tra goal khi xét node
        if not is_c2:
            if current.state == GOAL_STATE:
                steps.append(StepRecord(
                    current_node=current,
                    frontier_list=list(frontier),
                    reached_set=set(reached),
                    description="✅ ĐÃ TÌM THẤY GOAL!"
                ))
                return steps

        # Sinh con
        children = get_children(current)
        new_children = []
        goal_child = None

        for child in children:
            if child.state_tuple() not in reached:
                frontier.append(child)
                new_children.append(child)
                if is_c2 and child.state == GOAL_STATE:
                    goal_child = child
                    break

        if goal_child is not None:
            steps.append(StepRecord(
                current_node=current,
                frontier_list=list(frontier),
                reached_set=set(reached),
                description=f"Sinh {len(new_children)} node con → Phát hiện node con trùng khớp với ĐÍCH!"
            ))
            steps.append(StepRecord(
                current_node=goal_child,
                frontier_list=list(frontier),
                reached_set=set(reached),
                description="✅ ĐÃ TÌM THẤY GOAL khi sinh node con!"
            ))
            return steps

        steps.append(StepRecord(
            current_node=current,
            frontier_list=list(frontier),
            reached_set=set(reached),
            description=f"Sinh {len(new_children)} node con → thêm vào Frontier"
        ))

    if len(steps) >= max_steps:
        steps.append(StepRecord(
            current_node=None,
            frontier_list=[],
            reached_set=reached,
            description="⚠️ Đã đạt giới hạn bước mô phỏng."
        ))

    return steps


class PuzzleApp:
    def __init__(self, root):
        self.root = root
        self.root.title("8-Puzzle Simulator — BFS / DFS")
        self.root.configure(bg=BG_DARK)
        self.root.minsize(960, 820)
        self.root.resizable(True, True)

        self.initial_state = generate_random_state()
        self.steps = []
        self.current_step_idx = -1
        self.algorithm = tk.StringVar(value="BFS c1")

        self._build_ui()
        self._display_initial()

    def _build_ui(self):
        header = tk.Frame(self.root, bg=BG_CARD, pady=12)
        header.pack(fill=tk.X, padx=0)

        title_lbl = tk.Label(header, text="🧩  8-Puzzle Simulator",
                             font=("Segoe UI", 20, "bold"), fg=ACCENT_CYAN, bg=BG_CARD)
        title_lbl.pack(side=tk.LEFT, padx=24)

        algo_frame = tk.Frame(header, bg=BG_CARD)
        algo_frame.pack(side=tk.RIGHT, padx=24)

        tk.Label(algo_frame, text="Thuật toán:", font=("Segoe UI", 12),
                 fg=TEXT_SECONDARY, bg=BG_CARD).pack(side=tk.LEFT, padx=(0, 8))

        style = ttk.Style()
        style.theme_use('clam')
        style.configure("Algo.TRadiobutton", background=BG_CARD, foreground=TEXT_PRIMARY,
                        font=("Segoe UI", 12, "bold"), focuscolor=BG_CARD)
        style.map("Algo.TRadiobutton",
                  foreground=[('selected', ACCENT_CYAN), ('active', ACCENT_CYAN)])

        ttk.Radiobutton(algo_frame, text="BFS c1", variable=self.algorithm,
                        value="BFS c1", style="Algo.TRadiobutton",
                        command=self._on_algo_change).pack(side=tk.LEFT, padx=6)
        ttk.Radiobutton(algo_frame, text="BFS c2", variable=self.algorithm,
                        value="BFS c2", style="Algo.TRadiobutton",
                        command=self._on_algo_change).pack(side=tk.LEFT, padx=6)
        ttk.Radiobutton(algo_frame, text="DFS c1", variable=self.algorithm,
                        value="DFS c1", style="Algo.TRadiobutton",
                        command=self._on_algo_change).pack(side=tk.LEFT, padx=6)
        ttk.Radiobutton(algo_frame, text="DFS c2", variable=self.algorithm,
                        value="DFS c2", style="Algo.TRadiobutton",
                        command=self._on_algo_change).pack(side=tk.LEFT, padx=6)

        tk.Label(algo_frame, text="│", font=("Segoe UI", 14),
                 fg="#444466", bg=BG_CARD).pack(side=tk.LEFT, padx=4)

        style.configure("Greedy.TRadiobutton", background=BG_CARD,
                        foreground=ACCENT_ORANGE,
                        font=("Segoe UI", 12, "bold"), focuscolor=BG_CARD)
        style.map("Greedy.TRadiobutton",
                  foreground=[('selected', ACCENT_ORANGE), ('active', ACCENT_ORANGE)])

        ttk.Radiobutton(algo_frame, text="Greedy", variable=self.algorithm,
                        value="Greedy", style="Greedy.TRadiobutton",
                        command=self._on_algo_change).pack(side=tk.LEFT, padx=6)

        tk.Label(algo_frame, text="│", font=("Segoe UI", 14),
                 fg="#444466", bg=BG_CARD).pack(side=tk.LEFT, padx=4)

        style.configure("AStar.TRadiobutton", background=BG_CARD,
                        foreground=ACCENT_PURPLE,
                        font=("Segoe UI", 12, "bold"), focuscolor=BG_CARD)
        style.map("AStar.TRadiobutton",
                  foreground=[('selected', ACCENT_PURPLE), ('active', ACCENT_PURPLE)])

        ttk.Radiobutton(algo_frame, text="A*", variable=self.algorithm,
                        value="A*", style="AStar.TRadiobutton",
                        command=self._on_algo_change).pack(side=tk.LEFT, padx=6)

        tk.Label(algo_frame, text="│", font=("Segoe UI", 14),
                 fg="#444466", bg=BG_CARD).pack(side=tk.LEFT, padx=4)

        style.configure("IDS.TRadiobutton", background=BG_CARD,
                        foreground=ACCENT_PINK,
                        font=("Segoe UI", 12, "bold"), focuscolor=BG_CARD)
        style.map("IDS.TRadiobutton",
                  foreground=[('selected', ACCENT_PINK), ('active', ACCENT_PINK)])

        ttk.Radiobutton(algo_frame, text="IDS", variable=self.algorithm,
                        value="IDS", style="IDS.TRadiobutton",
                        command=self._on_algo_change).pack(side=tk.LEFT, padx=6)

        tk.Label(algo_frame, text="│", font=("Segoe UI", 14),
                 fg="#444466", bg=BG_CARD).pack(side=tk.LEFT, padx=4)

        style.configure("LDDG.TRadiobutton", background=BG_CARD,
                        foreground=ACCENT_GREEN,
                        font=("Segoe UI", 12, "bold"), focuscolor=BG_CARD)
        style.map("LDDG.TRadiobutton",
                  foreground=[('selected', ACCENT_GREEN), ('active', ACCENT_GREEN)])

        ttk.Radiobutton(algo_frame, text="LĐĐG", variable=self.algorithm,
                        value="LĐĐG", style="LDDG.TRadiobutton",
                        command=self._on_algo_change).pack(side=tk.LEFT, padx=6)

        tk.Label(algo_frame, text="│", font=("Segoe UI", 14),
                 fg="#444466", bg=BG_CARD).pack(side=tk.LEFT, padx=4)

        style.configure("LDDN.TRadiobutton", background=BG_CARD,
                        foreground=ACCENT_CYAN,
                        font=("Segoe UI", 12, "bold"), focuscolor=BG_CARD)
        style.map("LDDN.TRadiobutton",
                  foreground=[('selected', ACCENT_CYAN), ('active', ACCENT_CYAN)])

        ttk.Radiobutton(algo_frame, text="LĐDN", variable=self.algorithm,
                        value="LĐDN", style="LDDN.TRadiobutton",
                        command=self._on_algo_change).pack(side=tk.LEFT, padx=6)

        main_frame = tk.Frame(self.root, bg=BG_DARK)
        main_frame.pack(fill=tk.BOTH, expand=True, padx=20, pady=10)

        left_frame = tk.Frame(main_frame, bg=BG_DARK)
        left_frame.pack(side=tk.LEFT, fill=tk.Y, padx=(0, 10))

        self.step_label = tk.Label(left_frame, text="Bước: —",
                                   font=("Segoe UI", 13, "bold"),
                                   fg=ACCENT_ORANGE, bg=BG_DARK, anchor="w")
        self.step_label.pack(fill=tk.X, pady=(0, 4))

        self.desc_label = tk.Label(left_frame, text="",
                                   font=("Segoe UI", 11),
                                   fg=TEXT_SECONDARY, bg=BG_DARK, anchor="w",
                                   wraplength=320, justify=tk.LEFT)
        self.desc_label.pack(fill=tk.X, pady=(0, 8))

        self.puzzle_size = 300
        self.puzzle_canvas = tk.Canvas(left_frame, width=self.puzzle_size,
                                       height=self.puzzle_size,
                                       bg=BG_CARD, highlightthickness=2,
                                       highlightbackground=ACCENT_BLUE)
        self.puzzle_canvas.pack(pady=4)

        node_info_frame = tk.Frame(left_frame, bg=BG_CARD_ALT, padx=12, pady=8)
        node_info_frame.pack(fill=tk.X, pady=(10, 0))

        tk.Label(node_info_frame, text="NODE ĐANG XÉT",
                 font=("Segoe UI", 11, "bold"),
                 fg=ACCENT_PINK, bg=BG_CARD_ALT).pack(anchor="w")

        self.node_info_text = tk.Label(node_info_frame, text="(rỗng)",
                                       font=("Consolas", 10),
                                       fg=TEXT_PRIMARY, bg=BG_CARD_ALT,
                                       anchor="w", justify=tk.LEFT, wraplength=300)
        self.node_info_text.pack(anchor="w", pady=(2, 0))

        goal_frame = tk.Frame(left_frame, bg=BG_CARD_ALT, padx=12, pady=8)
        goal_frame.pack(fill=tk.X, pady=(10, 0))

        tk.Label(goal_frame, text="TRẠNG THÁI ĐÍCH",
                 font=("Segoe UI", 10, "bold"),
                 fg=ACCENT_GREEN, bg=BG_CARD_ALT).pack(anchor="w")

        self.goal_canvas = tk.Canvas(goal_frame, width=120, height=120,
                                     bg=BG_CARD_ALT, highlightthickness=0)
        self.goal_canvas.pack(pady=4)
        self._draw_mini_board(self.goal_canvas, GOAL_STATE, 120)

        right_frame = tk.Frame(main_frame, bg=BG_DARK)
        right_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

        frontier_card = tk.Frame(right_frame, bg=BG_CARD, padx=12, pady=8)
        frontier_card.pack(fill=tk.BOTH, expand=True, pady=(0, 6))

        frontier_header = tk.Frame(frontier_card, bg=BG_CARD)
        frontier_header.pack(fill=tk.X)

        tk.Label(frontier_header, text="📋  FRONTIER",
                 font=("Segoe UI", 12, "bold"),
                 fg=ACCENT_CYAN, bg=BG_CARD).pack(side=tk.LEFT)

        self.frontier_count_lbl = tk.Label(frontier_header, text="(0)",
                                           font=("Segoe UI", 11),
                                           fg=TEXT_SECONDARY, bg=BG_CARD)
        self.frontier_count_lbl.pack(side=tk.LEFT, padx=8)

        self.frontier_canvas = tk.Canvas(frontier_card, bg=BG_CARD,
                                          highlightthickness=0)
        frontier_scrollbar = tk.Scrollbar(frontier_card, orient=tk.VERTICAL,
                                          command=self.frontier_canvas.yview)
        self.frontier_inner = tk.Frame(self.frontier_canvas, bg=BG_CARD)

        self.frontier_inner.bind("<Configure>",
            lambda e: self.frontier_canvas.configure(scrollregion=self.frontier_canvas.bbox("all")))
        self.frontier_canvas.create_window((0, 0), window=self.frontier_inner, anchor="nw")
        self.frontier_canvas.configure(yscrollcommand=frontier_scrollbar.set)

        self.frontier_canvas.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, pady=4)
        frontier_scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

        self.frontier_canvas.bind("<Enter>", lambda e: self._bind_mousewheel(self.frontier_canvas))
        self.frontier_canvas.bind("<Leave>", lambda e: self._unbind_mousewheel())

        reached_card = tk.Frame(right_frame, bg=BG_CARD, padx=12, pady=8)
        reached_card.pack(fill=tk.BOTH, expand=True, pady=(6, 0))

        reached_header = tk.Frame(reached_card, bg=BG_CARD)
        reached_header.pack(fill=tk.X)

        tk.Label(reached_header, text="✅  REACHED",
                 font=("Segoe UI", 12, "bold"),
                 fg=ACCENT_GREEN, bg=BG_CARD).pack(side=tk.LEFT)

        self.reached_count_lbl = tk.Label(reached_header, text="(0)",
                                          font=("Segoe UI", 11),
                                          fg=TEXT_SECONDARY, bg=BG_CARD)
        self.reached_count_lbl.pack(side=tk.LEFT, padx=8)

        self.reached_canvas = tk.Canvas(reached_card, bg=BG_CARD,
                                         highlightthickness=0)
        reached_scrollbar = tk.Scrollbar(reached_card, orient=tk.VERTICAL,
                                         command=self.reached_canvas.yview)
        self.reached_inner = tk.Frame(self.reached_canvas, bg=BG_CARD)

        self.reached_inner.bind("<Configure>",
            lambda e: self.reached_canvas.configure(scrollregion=self.reached_canvas.bbox("all")))
        self.reached_canvas.create_window((0, 0), window=self.reached_inner, anchor="nw")
        self.reached_canvas.configure(yscrollcommand=reached_scrollbar.set)

        self.reached_canvas.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, pady=4)
        reached_scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

        self.reached_canvas.bind("<Enter>", lambda e: self._bind_mousewheel(self.reached_canvas))
        self.reached_canvas.bind("<Leave>", lambda e: self._unbind_mousewheel())

        btn_bar = tk.Frame(self.root, bg=BG_CARD, pady=12)
        btn_bar.pack(fill=tk.X, side=tk.BOTTOM)

        btn_style = {
            "font": ("Segoe UI", 12, "bold"),
            "bd": 0,
            "padx": 20,
            "pady": 8,
            "cursor": "hand2",
            "activeforeground": TEXT_BRIGHT,
        }

        self.btn_random = tk.Button(btn_bar, text="🎲  Random",
                                    bg=ACCENT_PURPLE, fg=TEXT_BRIGHT,
                                    activebackground="#9c4dff",
                                    command=self._on_random, **btn_style)
        self.btn_random.pack(side=tk.LEFT, padx=16)

        self.btn_manual = tk.Button(btn_bar, text="✏️  Nhập tay",
                                    bg="#5d4037", fg=TEXT_BRIGHT,
                                    activebackground="#8d6e63",
                                    command=self._on_manual_input, **btn_style)
        self.btn_manual.pack(side=tk.LEFT, padx=4)

        self.btn_solve = tk.Button(btn_bar, text="⚡  Giải",
                                   bg=ACCENT_BLUE, fg=TEXT_BRIGHT,
                                   activebackground="#1a4a8a",
                                   command=self._on_solve, **btn_style)
        self.btn_solve.pack(side=tk.LEFT, padx=4)

        self.btn_prev = tk.Button(btn_bar, text="◀  Trước",
                                  bg="#37474f", fg=TEXT_PRIMARY,
                                  activebackground="#546e7a",
                                  command=self._on_prev, state=tk.DISABLED,
                                  **btn_style)
        self.btn_prev.pack(side=tk.LEFT, padx=4)

        self.btn_next = tk.Button(btn_bar, text="Tiếp  ▶",
                                  bg=ACCENT_GREEN, fg="#1a1a2e",
                                  activebackground="#69f0ae",
                                  command=self._on_next, state=tk.DISABLED,
                                  **btn_style)
        self.btn_next.pack(side=tk.LEFT, padx=4)

        self.step_counter_lbl = tk.Label(btn_bar, text="",
                                         font=("Segoe UI", 11),
                                         fg=TEXT_SECONDARY, bg=BG_CARD)
        self.step_counter_lbl.pack(side=tk.RIGHT, padx=24)

    def _bind_mousewheel(self, canvas):
        self._active_canvas = canvas
        self.root.bind_all("<MouseWheel>", self._on_mousewheel)

    def _unbind_mousewheel(self):
        self.root.unbind_all("<MouseWheel>")

    def _on_mousewheel(self, event):
        if hasattr(self, '_active_canvas'):
            self._active_canvas.yview_scroll(int(-1 * (event.delta / 120)), "units")

    def _draw_board(self, canvas, state, size, highlight_action=None):
        canvas.delete("all")
        cell = size / 3
        pad = 4

        for i in range(3):
            for j in range(3):
                val = state[i][j]
                x0 = j * cell + pad
                y0 = i * cell + pad
                x1 = (j + 1) * cell - pad
                y1 = (i + 1) * cell - pad

                if val == 0:
                    canvas.create_rectangle(x0, y0, x1, y1,
                                            fill=BG_CARD, outline="#333355",
                                            width=2)
                else:
                    canvas.create_rectangle(x0, y0, x1, y1,
                                            fill=TILE_COLORS[val],
                                            outline="#222244", width=1)

                    canvas.create_text((x0 + x1) / 2, (y0 + y1) / 2,
                                       text=str(val),
                                       font=("Segoe UI", int(cell * 0.35), "bold"),
                                       fill=TILE_TEXT_COLORS[val])

    def _draw_mini_board(self, canvas, state, size):
        canvas.delete("all")
        cell = size / 3
        pad = 2
        for i in range(3):
            for j in range(3):
                val = state[i][j]
                x0 = j * cell + pad
                y0 = i * cell + pad
                x1 = (j + 1) * cell - pad
                y1 = (i + 1) * cell - pad

                if val == 0:
                    canvas.create_rectangle(x0, y0, x1, y1,
                                            fill=BG_CARD_ALT, outline="#333355")
                else:
                    canvas.create_rectangle(x0, y0, x1, y1,
                                            fill=TILE_COLORS[val], outline="")
                    canvas.create_text((x0 + x1) / 2, (y0 + y1) / 2,
                                       text=str(val),
                                       font=("Segoe UI", int(cell * 0.3), "bold"),
                                       fill=TILE_TEXT_COLORS[val])

    def _draw_node_card(self, parent, node, index=None, bg=BG_CARD):
        """Vẽ một thẻ node nhỏ gọn hiển thị trạng thái + thông tin."""
        algo = self.algorithm.get()
        is_greedy = 'Greedy' in algo
        is_astar  = 'A*' in algo
        is_ids    = 'IDS' in algo
        is_lddg   = 'LĐĐG' in algo
        is_lddn   = 'LĐDN' in algo

        if is_astar:
            border_color = ACCENT_PURPLE
        elif is_greedy:
            border_color = ACCENT_ORANGE
        elif is_ids:
            border_color = ACCENT_PINK
        elif is_lddg:
            border_color = ACCENT_GREEN
        elif is_lddn:
            border_color = ACCENT_CYAN
        else:
            border_color = "#333355"

        frame = tk.Frame(parent, bg=bg, padx=6, pady=4,
                         highlightbackground=border_color, highlightthickness=1)
        frame.pack(side=tk.LEFT, padx=3, pady=3)

        action_str = node.action if node.action else "Start"
        info = f"Act: {action_str}  |  g={node.depth}"
        tk.Label(frame, text=info, font=("Consolas", 8),
                 fg=TEXT_SECONDARY, bg=bg).pack(anchor="w")

        if (is_greedy or is_lddg or is_lddn) and hasattr(node, 'h_val'):
            h_color = ACCENT_GREEN if node.h_val == 0 else ACCENT_ORANGE
            tk.Label(frame, text=f"h = {node.h_val}",
                     font=("Consolas", 8, "bold"),
                     fg=h_color, bg=bg).pack(anchor="w")

        if is_astar and hasattr(node, 'f_val'):
            f_color = ACCENT_GREEN if node.h_val == 0 else ACCENT_PURPLE
            tk.Label(frame,
                     text=f"h={node.h_val}  g={node.depth}  f={node.f_val}",
                     font=("Consolas", 8, "bold"),
                     fg=f_color, bg=bg).pack(anchor="w")

        mini = tk.Canvas(frame, width=78, height=78, bg=bg, highlightthickness=0)
        mini.pack()
        self._draw_mini_board(mini, node.state, 78)

        return frame

    def _draw_state_card(self, parent, state_tuple, bg=BG_CARD):
        """Vẽ thẻ nhỏ cho reached (chỉ hiển thị trạng thái)."""
        frame = tk.Frame(parent, bg=bg, padx=4, pady=4,
                         highlightbackground="#2e4a2e", highlightthickness=1)
        frame.pack(side=tk.LEFT, padx=2, pady=2)

        mini = tk.Canvas(frame, width=60, height=60, bg=bg, highlightthickness=0)
        mini.pack()
        state = [list(state_tuple[i]) for i in range(3)]
        self._draw_mini_board(mini, state, 60)

        return frame

    def _display_initial(self):
        self._draw_board(self.puzzle_canvas, self.initial_state, self.puzzle_size)
        self.step_label.config(text="Bước: — (chưa giải)")
        self.desc_label.config(text="Nhấn '⚡ Giải' để bắt đầu mô phỏng thuật toán.")
        self.node_info_text.config(text="(rỗng)")
        self._clear_panel(self.frontier_inner)
        self._clear_panel(self.reached_inner)
        self.frontier_count_lbl.config(text="(0)")
        self.reached_count_lbl.config(text="(0)")
        self.step_counter_lbl.config(text="")
        self.btn_prev.config(state=tk.DISABLED)
        self.btn_next.config(state=tk.DISABLED)

    def _display_step(self, idx):
        if idx < 0 or idx >= len(self.steps):
            return

        step = self.steps[idx]
        self.current_step_idx = idx

        self.step_label.config(text=f"Bước: {idx} / {len(self.steps) - 1}")
        self.desc_label.config(text=step.description)
        self.step_counter_lbl.config(text=f"Step {idx}/{len(self.steps)-1}")

        if step.current_node:
            self._draw_board(self.puzzle_canvas, step.current_node.state, self.puzzle_size)
            action_str = step.current_node.action if step.current_node.action else "—"
            self.node_info_text.config(
                text=f"Trạng thái: {self._state_str(step.current_node.state)}\n"
                     f"Hành động:  {action_str}\n"
                     f"Số bước:    {step.current_node.depth}")
        else:
            self._draw_board(self.puzzle_canvas, self.initial_state, self.puzzle_size)
            self.node_info_text.config(text="(rỗng)")

        self._clear_panel(self.frontier_inner)
        displayed = min(len(step.frontier_list), 50) 
        self.frontier_count_lbl.config(text=f"({len(step.frontier_list)})")

        row_frame = None
        for i, node in enumerate(step.frontier_list[:displayed]):
            if i % 5 == 0:
                row_frame = tk.Frame(self.frontier_inner, bg=BG_CARD)
                row_frame.pack(fill=tk.X, anchor="w")
            self._draw_node_card(row_frame, node, i)

        if len(step.frontier_list) > displayed:
            tk.Label(self.frontier_inner,
                     text=f"... và {len(step.frontier_list) - displayed} node khác",
                     font=("Segoe UI", 9, "italic"),
                     fg=TEXT_SECONDARY, bg=BG_CARD).pack(anchor="w", padx=8, pady=4)

        self._clear_panel(self.reached_inner)
        reached_list = list(step.reached_set)
        displayed_r = min(len(reached_list), 60)
        self.reached_count_lbl.config(text=f"({len(reached_list)})")

        row_frame_r = None
        for i, st in enumerate(reached_list[:displayed_r]):
            if i % 7 == 0:
                row_frame_r = tk.Frame(self.reached_inner, bg=BG_CARD)
                row_frame_r.pack(fill=tk.X, anchor="w")
            self._draw_state_card(row_frame_r, st)

        if len(reached_list) > displayed_r:
            tk.Label(self.reached_inner,
                     text=f"... và {len(reached_list) - displayed_r} trạng thái khác",
                     font=("Segoe UI", 9, "italic"),
                     fg=TEXT_SECONDARY, bg=BG_CARD).pack(anchor="w", padx=8, pady=4)

        self.btn_prev.config(state=tk.NORMAL if idx > 0 else tk.DISABLED)
        self.btn_next.config(state=tk.NORMAL if idx < len(self.steps) - 1 else tk.DISABLED)

    def _state_str(self, state):
        return " | ".join([" ".join(str(x) for x in row) for row in state])

    def _clear_panel(self, panel):
        for widget in panel.winfo_children():
            widget.destroy()
    def _on_algo_change(self):
        self.steps = []
        self.current_step_idx = -1
        self._display_initial()

    def _on_manual_input(self):
        """Mở dialog nhập tay trạng thái ban đầu 3x3."""
        dialog = tk.Toplevel(self.root)
        dialog.title("Nhập trạng thái ban đầu")
        dialog.configure(bg=BG_DARK)
        dialog.resizable(False, False)
        dialog.grab_set()  # Modal

        dw, dh = 400, 530
        sw = self.root.winfo_screenwidth()
        sh = self.root.winfo_screenheight()
        dialog.geometry(f"{dw}x{dh}+{(sw-dw)//2}+{(sh-dh)//2}")

        tk.Label(dialog, text="✏️  Nhập trạng thái ban đầu",
                 font=("Segoe UI", 14, "bold"),
                 fg=ACCENT_CYAN, bg=BG_DARK).pack(pady=(14, 4))

        tk.Label(dialog,
                 text="Nhập số từ 0–8 (0 = ô trống), mỗi số xuất hiện đúng 1 lần.",
                 font=("Segoe UI", 9), fg=TEXT_SECONDARY, bg=BG_DARK,
                 wraplength=340).pack(pady=(0, 12))

        grid_frame = tk.Frame(dialog, bg=BG_CARD, padx=14, pady=12)
        grid_frame.pack(padx=24)

        entries = []
        for i in range(3):
            row_entries = []
            for j in range(3):
                vcmd = (dialog.register(lambda P: P == '' or (P.isdigit() and len(P) == 1)), '%P')
                e = tk.Entry(grid_frame, width=4,
                             font=("Segoe UI", 22, "bold"),
                             fg=TEXT_BRIGHT, bg=ACCENT_BLUE,
                             insertbackground=ACCENT_CYAN,
                             justify=tk.CENTER, bd=0,
                             highlightthickness=2,
                             highlightcolor=ACCENT_CYAN,
                             highlightbackground="#333355",
                             validate='key', validatecommand=vcmd)
                e.grid(row=i, column=j, padx=6, pady=5, ipady=8)
                def make_advance(r, c):
                    def advance(event):
                        if event.char.isdigit():
                            next_r, next_c = (r, c + 1) if c < 2 else (r + 1, 0)
                            if next_r < 3:
                                entries[next_r][next_c].focus_set()
                    return advance
                e.bind('<KeyRelease>', make_advance(i, j))
                row_entries.append(e)
            entries.append(row_entries)

        for i in range(3):
            for j in range(3):
                entries[i][j].insert(0, str(self.initial_state[i][j]))

        entries[0][0].focus_set()

        err_lbl = tk.Label(dialog, text="", font=("Segoe UI", 10),
                           fg=ACCENT_PINK, bg=BG_DARK)
        err_lbl.pack(pady=(6, 0))

        def validate_and_apply():
            try:
                values = []
                for i in range(3):
                    for j in range(3):
                        txt = entries[i][j].get().strip()
                        if txt == '':
                            err_lbl.config(text=f"⚠ Ô [{i+1},{j+1}] còn trống!")
                            return
                        v = int(txt)
                        if v < 0 or v > 8:
                            err_lbl.config(text=f"⚠ Giá trị phải từ 0 đến 8!")
                            return
                        values.append(v)

                if len(set(values)) != 9:
                    err_lbl.config(text="⚠ Các số phải khác nhau (0–8 mỗi số 1 lần)!")
                    return

                state = [values[i*3:(i+1)*3] for i in range(3)]

                if state == GOAL_STATE:
                    err_lbl.config(text="⚠ Đây đã là trạng thái đích rồi!")
                    return

                if not is_solvable(state):
                    err_lbl.config(text="⚠ Trạng thái này không thể giải được!")
                    return

                self.initial_state = state
                self.steps = []
                self.current_step_idx = -1
                self._display_initial()
                dialog.destroy()

            except ValueError:
                err_lbl.config(text="⚠ Chỉ nhập số nguyên!")

        btn_frame = tk.Frame(dialog, bg=BG_DARK)
        btn_frame.pack(pady=12)

        tk.Button(btn_frame, text="✅  Áp dụng",
                  font=("Segoe UI", 12, "bold"),
                  bg=ACCENT_GREEN, fg="#1a1a2e",
                  activebackground="#69f0ae",
                  bd=0, padx=18, pady=8, cursor="hand2",
                  command=validate_and_apply).pack(side=tk.LEFT, padx=8)

        tk.Button(btn_frame, text="❌  Huỷ",
                  font=("Segoe UI", 12, "bold"),
                  bg="#37474f", fg=TEXT_PRIMARY,
                  activebackground="#546e7a",
                  bd=0, padx=18, pady=8, cursor="hand2",
                  command=dialog.destroy).pack(side=tk.LEFT, padx=8)

        dialog.bind('<Return>', lambda e: validate_and_apply())
        dialog.bind('<Escape>', lambda e: dialog.destroy())

    def _on_random(self):
        self.initial_state = generate_random_state()
        self.steps = []
        self.current_step_idx = -1
        self._display_initial()

    def _on_solve(self):
        algo = self.algorithm.get()
        self.root.config(cursor="wait")
        self.root.update()

        try:
            self.steps = solve_steps(self.initial_state, algo)
            self.current_step_idx = 0
            self._display_step(0)
        except Exception as e:
            messagebox.showerror("Lỗi", f"Lỗi khi giải: {e}")
        finally:
            self.root.config(cursor="")

    def _on_prev(self):
        if self.current_step_idx > 0:
            self._display_step(self.current_step_idx - 1)

    def _on_next(self):
        if self.current_step_idx < len(self.steps) - 1:
            self._display_step(self.current_step_idx + 1)


if __name__ == "__main__":
    root = tk.Tk()
    win_w, win_h = 1050, 860
    sw = root.winfo_screenwidth()
    sh = root.winfo_screenheight()
    x = (sw - win_w) // 2
    y = (sh - win_h) // 2
    root.geometry(f"{win_w}x{win_h}+{x}+{y}")

    app = PuzzleApp(root)
    root.mainloop()
